<a href="https://colab.research.google.com/github/mirikrupkin/structural-ai-validation-suite/blob/main/structural_validation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 0. ENVIRONMENT SETUP
# Automatically checks for and installs required dependencies
# ==========================================
print("[INFO] Checking environment dependencies...")
!pip install biopython py3Dmol pandas -q
print("[INFO] Environment is ready! All dependencies met.")

In [ ]:
%%writefile src/fetcher.py
# ==========================================
# API MODULE: fetcher.py
# 1. Handles AlphaFold API and PDB downloads
# ==========================================
import json
from urllib.request import urlopen

def fetch_alphafold_prediction(uniprot_id):
    """Fetch AlphaFold prediction metadata and PDB download link via API."""
    url = f"https://alphafold.com/api/prediction/{uniprot_id}"
    try:
        with urlopen(url) as response:
            data = json.loads(response.read().decode())
            if isinstance(data, list) and len(data) > 0:
                entry = data[0]
                pdb_url = entry.get('pdbUrl')
                protein_name = entry.get('uniprotDescription', f'AlphaFold Model ({uniprot_id})')
                return pdb_url, protein_name
    except Exception as e:
        print(f"Error fetching AlphaFold data for UniProt {uniprot_id}: {e}")
    return None, f"UniProt Model ({uniprot_id})"

def download_file(url, filename):
    """Download any file locally from a URL."""
    with urlopen(url) as response, open(filename, 'wb') as f:
        f.write(response.read())

In [ ]:
%%writefile src/alignment.py
# ==========================================
# 2. Build the Core Module (alignment.py)
# ==========================================
import os
import pandas as pd
from Bio.PDB import PDBParser, Superimposer, PDBIO
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.Align import PairwiseAligner

d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K', 'ILE': 'I',
         'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 'GLY': 'G', 'HIS': 'H',
         'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 'ALA': 'A', 'VAL': 'V', 'GLU': 'E',
         'TYR': 'Y', 'MET': 'M'}

def parse_structure_file(filepath, structure_id="model"):
    ext = os.path.splitext(filepath)[1].lower()
    parser = MMCIFParser(QUIET=True) if ext == ".cif" else PDBParser(QUIET=True)
    return parser.get_structure(structure_id, filepath)

def calculate_structural_rmsd_and_deviation(ref_pdb_path, target_pdb_path, trim_excess=False):
    ref_structure = parse_structure_file(ref_pdb_path, "experimental")
    target_structure = parse_structure_file(target_pdb_path, "predicted")

    ref_chains = sorted([c for m in ref_structure for c in m], key=lambda c: len([r for r in c if 'CA' in r]), reverse=True)
    target_chains = sorted([c for m in target_structure for c in m], key=lambda c: len([r for r in c if 'CA' in r]), reverse=True)

    ref_atoms_matched = []
    target_atoms_matched = []

    aligner = PairwiseAligner()
    aligner.mode = 'local'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    aligner.match_score = 1
    aligner.mismatch_score = -2

    for r_chain, t_chain in zip(ref_chains, target_chains):
        r_res = [res for res in r_chain if 'CA' in res]
        t_res = [res for res in t_chain if 'CA' in res]

        if not r_res or not t_res:
            continue

        r_seq = "".join([d3to1.get(res.get_resname(), 'X') for res in r_res])
        t_seq = "".join([d3to1.get(res.get_resname(), 'X') for res in t_res])

        alignment = aligner.align(r_seq, t_seq)[0]
        r_indices = alignment.aligned[0]
        t_indices = alignment.aligned[1]

        for (r_start, r_end), (t_start, t_end) in zip(r_indices, t_indices):
            for i in range(r_end - r_start):
                ref_atoms_matched.append(r_res[r_start + i]['CA'])
                target_atoms_matched.append(t_res[t_start + i]['CA'])

    min_len = len(ref_atoms_matched)
    identical_count = sum(1 for r, t in zip(ref_atoms_matched, target_atoms_matched)
                          if r.get_parent().get_resname() == t.get_parent().get_resname())

    aligned_ref_filename = None
    if trim_excess:
        # Trim Target (AlphaFold)
        matched_target_ids = {atom.get_parent().get_full_id() for atom in target_atoms_matched}
        for model in target_structure:
            for chain in model:
                res_to_detach = [res.get_id() for res in chain if res.has_id('CA') and res.get_full_id() not in matched_target_ids]
                for res_id in res_to_detach:
                    chain.detach_child(res_id)

        # Trim Reference & STRIP HETATM (drugs, waters, ions) so zoom bounds match perfectly!
        matched_ref_ids = {atom.get_parent().get_full_id() for atom in ref_atoms_matched}
        for model in ref_structure:
            for chain in model:
                res_to_detach = []
                for res in chain:
                    # Detach if not part of matched chain OR if it's a heteroatom (waters, ligands, ions)
                    if res.id[0].strip() != '' or (res.has_id('CA') and res.get_full_id() not in matched_ref_ids):
                        res_to_detach.append(res.get_id())
                for res_id in res_to_detach:
                    chain.detach_child(res_id)

        trimmed_info = "Construct Trimming & Heteroatom Stripping: Applied successfully."

        aligned_ref_filename = "aligned_reference.pdb"
        io_ref = PDBIO()
        io_ref.set_structure(ref_structure)
        io_ref.save(aligned_ref_filename)
    else:
        trimmed_info = "No trimming applied (Full-length alignment)."

    deviation_records = []
    for r_atom, t_atom in zip(ref_atoms_matched, target_atoms_matched):
        diff_vector = r_atom.get_coord() - t_atom.get_coord()
        distance = float((diff_vector ** 2).sum() ** 0.5)
        res = t_atom.get_parent()
        deviation_records.append({
            "Residue_ID": res.get_id()[1],
            "Residue": res.get_resname(),
            "Structural_Deviation_Angstroms": round(distance, 3)
        })

    seq_identity = (identical_count / min_len) * 100 if min_len > 0 else 0

    superimposer = Superimposer()
    superimposer.set_atoms(ref_atoms_matched, target_atoms_matched)
    superimposer.apply(target_structure.get_atoms())

    aligned_filename = "aligned_prediction.pdb"
    io = PDBIO()
    io.set_structure(target_structure)
    io.save(aligned_filename)

    df_dev = pd.DataFrame(deviation_records)
    return superimposer.rms, min_len, seq_identity, aligned_filename, aligned_ref_filename, df_dev, trimmed_info


In [ ]:
%%writefile src/metrics.py
# ==========================================
# Build the Analytics Module (metrics.py)
# ==========================================
import pandas as pd
from Bio.PDB import ShrakeRupley
from src.alignment import parse_structure_file

def analyze_alphafold_model(pdb_path):
    """Parse structure (PDB or CIF), compute SASA, and extract pLDDT from B-factors."""
    structure = parse_structure_file(pdb_path, "af_model")

    sr = ShrakeRupley()
    sr.compute(structure, level="R")

    records = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    records.append({
                        "Residue_ID": residue.get_id()[1],
                        "Residue": residue.get_resname(),
                        "pLDDT": residue["CA"].get_bfactor(),
                        "SASA": residue.sasa
                    })
    return pd.DataFrame(records)

In [ ]:
import os
import pandas as pd
from IPython.display import display, HTML

from src.fetcher import fetch_alphafold_prediction, download_file
from src.alignment import calculate_structural_rmsd_and_deviation
from src.metrics import analyze_alphafold_model

print("==================================================")
print("   STRUCTURAL INTEGRITY & VALIDATION PIPELINE     ")
print("==================================================")
print("Select a Validation Mode:")
print("  [1] 🧬 HIV-1 RTase (Multi-chain asymmetry)")
print("  [2] 🐣 Lysozyme C (Signal peptide cleavage)")
print("  [3] ⛓️  Polyubiquitin-C (Repeat domain extraction)")
print("  [4] 🎯 KRAS Oncology Target (Crystal dimer isolation)")
print("  [5] 🧪 Custom Mode (Enter your own UniProt/PDB)")

raw_choice = input("\nEnter choice [1-5] or type any UniProt ID directly: ").strip().upper()

# SMART ROUTING LOGIC
if raw_choice in ['1', '2', '3', '4', '5']:
    choice = raw_choice
elif raw_choice:
    choice = '5'
    custom_uniprot_override = raw_choice
else:
    choice = '1'

target_filename = None
target_name = None
exp_filename = None
should_trim = False
aligned_pred_file = None
aligned_ref_file = None
uniprot_id = ""

if choice == '2':
    print("\n[LYSOZYME CASE STUDY] Running automated pipeline...")
    uniprot_id = "P00698"
    print("--> Step 1: Fetching AlphaFold model for Lysozyme C (P00698)...")
    af_url, protein_title = fetch_alphafold_prediction(uniprot_id)
    target_name = protein_title
    if af_url:
        target_filename = f"AF_{uniprot_id}.pdb"
        download_file(af_url, target_filename)

    exp_filename = "EXP_1IEE.pdb"
    print("--> Step 2: Auto-selecting Experimental Reference PDB (1IEE)...")
    download_file("https://files.rcsb.org/download/1IEE.pdb", exp_filename)
    should_trim = True

elif choice == '3':
    print("\n[POLYUBIQUITIN CASE STUDY] Running automated pipeline...")
    uniprot_id = "P0CG48"
    print("--> Step 1: Fetching AlphaFold model for Polyubiquitin-C (P0CG48)...")
    af_url, protein_title = fetch_alphafold_prediction(uniprot_id)
    target_name = protein_title
    if af_url:
        target_filename = f"AF_{uniprot_id}.pdb"
        download_file(af_url, target_filename)

    exp_filename = "EXP_1UBQ.pdb"
    print("--> Step 2: Auto-selecting Experimental Reference PDB (1UBQ)...")
    download_file("https://files.rcsb.org/download/1UBQ.pdb", exp_filename)
    should_trim = True

elif choice == '4':
    print("\n[KRAS ONCOLOGY CASE STUDY] Running automated pipeline...")
    uniprot_id = "P01116"
    print("--> Step 1: Fetching AlphaFold model for GTPase KRas (P01116)...")
    af_url, protein_title = fetch_alphafold_prediction(uniprot_id)
    target_name = protein_title
    if af_url:
        target_filename = f"AF_{uniprot_id}.pdb"
        download_file(af_url, target_filename)

    exp_filename = "EXP_4OBE.pdb"
    print("--> Step 2: Auto-selecting Experimental Reference PDB (4OBE)...")
    download_file("https://files.rcsb.org/download/4OBE.pdb", exp_filename)
    should_trim = True

elif choice == '5':
    print("\n[CUSTOM MODE ACTIVATED]")
    uniprot_id = locals().get('custom_uniprot_override') or input("Enter AlphaFold UniProt ID: ").strip().upper()
    if not uniprot_id:
        print("No ID entered. Defaulting to P00698...")
        uniprot_id = "P00698"

    print(f"--> Fetching model for UniProt: {uniprot_id}...")
    af_url, protein_title = fetch_alphafold_prediction(uniprot_id)
    target_name = protein_title
    if af_url:
        target_filename = f"AF_{uniprot_id}.pdb"
        download_file(af_url, target_filename)

    pdb_input = input("\nEnter 4-letter PDB code for experimental comparison (Press Enter to skip): ").strip().upper()
    if pdb_input and pdb_input not in ['N', 'NO', 'NONE']:
        if len(pdb_input) == 4:
            exp_filename = f"EXP_{pdb_input}.pdb"
            print(f"--> Downloading PDB {pdb_input}...")
            download_file(f"https://files.rcsb.org/download/{pdb_input}.pdb", exp_filename)
            should_trim = input("\nTrim excess prediction tails & isolate PDB chain? [y/N]: ").strip().lower() in ['y', 'yes']

else:
    # Option 1 (RTase Demo)
    print("\n[RTASE DEMO ACTIVATED] Running automated pipeline...")
    target_name = "HIV-1 RT Heterodimer"
    target_filename = "1rev_prediction.cif"
    print(f"--> Step 1: Downloading pre-computed prediction for {target_name}...")
    print("    [DEMO NOTE] This custom prediction file was generated by extracting the")
    print("    experimental sequence from PDB 1REV and running it through AlphaFold.")
    download_file("https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/1rev_prediction.cif", target_filename)

    exp_filename = "EXP_1REV.pdb"
    print("--> Step 2: Auto-selecting Experimental Reference PDB (1REV)...")
    download_file("https://files.rcsb.org/download/1REV.pdb", exp_filename)
    should_trim = True

# --- Explicit Configuration Summary ---
print("\n" + "-"*45)
print("⚙️  PIPELINE CONFIGURATION SUMMARY")
print(f"   Prediction Used: {target_name} ({target_filename})")
if exp_filename:
    print(f"   Reference PDB Used: {exp_filename}")
    print(f"   Construct Trimming: {'Enabled (Target & Reference)' if should_trim else 'Disabled'}")
else:
    print("   Reference PDB Used: None Selected")
print("-"*45)

# --- Pipeline Execution ---
print(f"\n--> Analyzing target model metrics...")
df_metrics = analyze_alphafold_model(target_filename)

if exp_filename and os.path.exists(exp_filename):
    print("--> Running chain-aware spatial alignment (Smith-Waterman local mapping)...")

    rmsd_val, matched_res, seq_id, aligned_pred_file, aligned_ref_file, df_dev, trim_report = calculate_structural_rmsd_and_deviation(
        exp_filename, target_filename, trim_excess=should_trim
    )

    print("\n--- Benchmark Results ---")
    print(f"Matched Alpha-Carbon Residues: {matched_res}")
    print(f"Sequence Identity: {seq_id:.1f}%")
    print(f"Global Coordinate RMSD: {rmsd_val:.3f} Å")
    print(f"Construct Status: {trim_report}")

    # 💡 BIOLOGICAL INSIGHTS
    if choice == '2' and should_trim:
        print("\n💡 [BIOLOGICAL INSIGHT] Notice the 18 residues pruned from the AlphaFold model!")
        print("   Lysozyme C contains an 18-amino acid N-terminal signal peptide that gets")
        print("   cleaved in vivo. AlphaFold predicts the raw sequence, but the crystal (1IEE)")
        print("   captures the mature protein. The algorithm perfectly handles this discrepancy.")

    elif choice == '3' and should_trim:
        print("\n💡 [BIOLOGICAL INSIGHT] Polyubiquitin-C is synthesized as a massive continuous chain")
        print("   of 9 repeat domains! AlphaFold predicts the entire ~685 residue chain.")
        print("   However, the crystal (1UBQ) is just a single 76-residue monomer.")
        print("   The Smith-Waterman local alignment perfectly located the matching repeat")
        print("   and dynamically pruned away over 600 unaligned residues!")

    elif choice == '4' and should_trim:
        print("\n💡 [BIOLOGICAL INSIGHT] KRAS is a premier oncology drug target.")
        print("   The experimental crystal structure (4OBE) crystallized as a dimer in the")
        print("   asymmetric unit, whereas AlphaFold predicts a single monomer. My chain-aware")
        print("   Smith-Waterman algorithm successfully isolated the matching monomer and pruned")
        print("   the extra crystallography artifacts for a pristine 1-to-1 comparison.")

    elif choice == '1':
        print("\n💡 [BIOLOGICAL INSIGHT] HIV-1 Reverse Transcriptase is a heterodimer (p66/p51).")
        print("   Fascinatingly, the p51 subunit is just a truncated version of the p66 sequence,")
        print("   yet it folds into a completely different, compact 3D conformation! This pipeline")
        print("   successfully maps multi-chain asymmetry without breaking the alignment.")

# --- 3D Visualization ---
print("\n[INFO] Rendering synchronized side-by-side 3D cartoon view...")
try:
    import py3Dmol
    viewer = py3Dmol.view(viewergrid=(1, 2), width=920, height=450)

    left_file = aligned_pred_file if (aligned_pred_file and os.path.exists(aligned_pred_file)) else target_filename
    with open(left_file, 'r') as f:
        viewer.addModel(f.read(), "pdb" if left_file.endswith(".pdb") else "cif", viewer=(0, 0))
    viewer.setStyle({'viewer': (0, 0)}, {'cartoon': {'color': 'lightblue'}})

    right_file = aligned_ref_file if (aligned_ref_file and os.path.exists(aligned_ref_file)) else exp_filename
    if right_file and os.path.exists(right_file):
        with open(right_file, 'r') as f:
            viewer.addModel(f.read(), 'pdb', viewer=(0, 1))
        viewer.setStyle({'viewer': (0, 1)}, {'cartoon': {'color': 'cyan'}})

    viewer.zoomTo()
    viewer.show()
except Exception as e:
    print(f"[WARNING] Visualization error: {e}")

print("\n" + "="*50)
print("🎉 [EXECUTION COMPLETE]")
print("👉 What's next? Rerun the cell to explore another mode.")
print("   Try options 1 through 4 to see how the algorithm handles")
print("   different structural bioinformatics edge cases!")
print("="*50 + "\n")